In [1]:
import os, glob, zipfile
import numpy as np
import pandas as pd

ZIP_PATH = "/content/datathon-2026-round-1.zip"
EXTRACT_DIR = "/content/datathon_data"

if os.path.exists(ZIP_PATH) and not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)

sales_candidates = glob.glob("/content/**/sales.csv", recursive=True)
sample_candidates = glob.glob("/content/**/sample_submission.csv", recursive=True)

print("sales candidates:", sales_candidates)
print("sample candidates:", sample_candidates)

assert len(sales_candidates) > 0, "Không tìm thấy sales.csv"
assert len(sample_candidates) > 0, "Không tìm thấy sample_submission.csv"

SALES_PATH = sales_candidates[0]
SUB_PATH = sample_candidates[0]

sales = pd.read_csv(SALES_PATH)
sample = pd.read_csv(SUB_PATH)

sales["Date"] = pd.to_datetime(sales["Date"])
sample["Date"] = pd.to_datetime(sample["Date"])

sales = sales.sort_values("Date").reset_index(drop=True)
sample = sample.sort_values("Date").reset_index(drop=True)

print("Loaded sales:", sales.shape, SALES_PATH)
print("Loaded sample:", sample.shape, SUB_PATH)
display(sales.head())
display(sample.head())

sales candidates: ['/content/datathon_data/sales.csv']
sample candidates: ['/content/datathon_data/sample_submission.csv']
Loaded sales: (3833, 3) /content/datathon_data/sales.csv
Loaded sample: (548, 3) /content/datathon_data/sample_submission.csv


,Date,Revenue,COGS
0,2012-07-04,5123547.94,3982991.19
1,2012-07-05,2751773.45,2150580.23
2,2012-07-06,3054029.42,2517632.84
3,2012-07-07,2667930.94,2108246.62
4,2012-07-08,2360851.90,1808622.79


,Date,Revenue,COGS
0,2023-01-01,2665507.20,2518885.15
1,2023-01-02,1280007.89,1136463.00
2,2023-01-03,1015899.51,822721.12
3,2023-01-04,1142997.27,914554.18
4,2023-01-05,1236312.34,984390.24


In [2]:
import os, glob

candidates = glob.glob("/content/**/sales.csv", recursive=True)
print("Found sales.csv:", candidates)

assert len(candidates) > 0, "Không tìm thấy sales.csv sau khi unzip"

SALES_PATH = candidates[0]
DATA_DIR = os.path.dirname(SALES_PATH)

print("DATA_DIR =", DATA_DIR)
print("SALES_PATH =", SALES_PATH)

print("\nFiles in DATA_DIR:")
print(os.listdir(DATA_DIR))

Found sales.csv: ['/content/datathon_data/sales.csv']
DATA_DIR = /content/datathon_data
SALES_PATH = /content/datathon_data/sales.csv

Files in DATA_DIR:
['promotions.csv', 'products.csv', 'order_items.csv', 'orders.csv', 'reviews.csv', 'returns.csv', 'baseline.ipynb', 'geography.csv', 'sales.csv', 'shipments.csv', 'inventory.csv', 'web_traffic.csv', 'sample_submission.csv', 'customers.csv', 'payments.csv']


In [3]:
import os

required_files = [
    "sales.csv",
    "sample_submission.csv",
    "orders.csv",
    "order_items.csv",
    "products.csv",
    "payments.csv",
    "shipments.csv",
    "returns.csv",
    "web_traffic.csv",
    "inventory.csv"
]

for f in required_files:
    path = os.path.join(DATA_DIR, f)
    print(f"{f:25s}", "OK" if os.path.exists(path) else "MISSING")

sales.csv                 OK
sample_submission.csv     OK
orders.csv                OK
order_items.csv           OK
products.csv              OK
payments.csv              OK
shipments.csv             OK
returns.csv               OK
web_traffic.csv           OK
inventory.csv             OK


In [4]:
def clean_sales_strong(df):
    df = df.copy()
    df = df[["Date", "Revenue", "COGS"]]
    df = df.groupby("Date", as_index=False)[["Revenue", "COGS"]].sum()

    full_dates = pd.date_range(df["Date"].min(), df["Date"].max(), freq="D")
    df = df.set_index("Date").reindex(full_dates)
    df.index.name = "Date"

    for col in ["Revenue", "COGS"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].interpolate(limit_direction="both")
        df[col] = df[col].fillna(df[col].median())

        roll_med = df[col].rolling(45, center=True, min_periods=10).median()
        resid = df[col] - roll_med
        mad = resid.abs().rolling(45, center=True, min_periods=10).median()

        lower = roll_med - 7.5 * mad
        upper = roll_med + 7.5 * mad

        mask = mad.notna() & (mad > 1)
        df.loc[mask, col] = np.minimum(
            np.maximum(df.loc[mask, col], lower.loc[mask]),
            upper.loc[mask]
        )

    df = df.reset_index()
    return df

sales_clean = clean_sales_strong(sales)

tmp = sales_clean.copy()
tmp["year"] = tmp["Date"].dt.year
yearly = tmp.groupby("year")[["Revenue", "COGS"]].sum()
yearly["Revenue_yoy"] = yearly["Revenue"].pct_change()
yearly["COGS_yoy"] = yearly["COGS"].pct_change()

display(yearly)

,Revenue,COGS,Revenue_yoy,COGS_yoy
year,,,,
2012,7.359801e+08,5.834135e+08,NaN,NaN
2013,1.657062e+09,1.464950e+09,1.251503,1.510998
2014,1.859707e+09,1.565587e+09,0.122292,0.068696
2015,1.883166e+09,1.660417e+09,0.012614,0.060572
2016,2.093702e+09,1.773155e+09,0.111799,0.067897
2017,1.909995e+09,1.690361e+09,-0.087743,-0.046693
2018,1.848489e+09,1.538761e+09,-0.032202,-0.089685
2019,1.136801e+09,1.003681e+09,-0.385010,-0.347734
2020,1.048782e+09,8.825002e+08,-0.077427,-0.120736


In [5]:
import numpy as np
import pandas as pd
import random
import os
from sklearn.metrics import mean_squared_error

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)

def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

sales = sales.copy()
sample = sample.copy()

sales["Date"] = pd.to_datetime(sales["Date"])
sample["Date"] = pd.to_datetime(sample["Date"])

sales = sales.sort_values("Date").reset_index(drop=True)
sample = sample.sort_values("Date").reset_index(drop=True)

print(sales.head())
print(sales.tail())
print(sample.head())
print(sample.tail())

        Date     Revenue        COGS
0 2012-07-04  5123547.94  3982991.19
1 2012-07-05  2751773.45  2150580.23
2 2012-07-06  3054029.42  2517632.84
3 2012-07-07  2667930.94  2108246.62
4 2012-07-08  2360851.90  1808622.79
           Date     Revenue        COGS
3828 2022-12-27  2100553.66  2184872.24
3829 2022-12-28  3448729.20  3513621.00
3830 2022-12-29  3083944.33  3170787.10
3831 2022-12-30  2884668.76  3022292.15
3832 2022-12-31  2383037.48  2279288.13
        Date     Revenue        COGS
0 2023-01-01  2665507.20  2518885.15
1 2023-01-02  1280007.89  1136463.00
2 2023-01-03  1015899.51   822721.12
3 2023-01-04  1142997.27   914554.18
4 2023-01-05  1236312.34   984390.24
          Date     Revenue        COGS
543 2024-06-27  4111585.95  4048492.76
544 2024-06-28  4591751.39  4504603.05
545 2024-06-29  6582807.93  6478608.03
546 2024-06-30  5484749.90  5386065.13
547 2024-07-01  5057610.99  4989753.02


In [6]:
def get_days_in_year(y):
    return 366 if pd.Timestamp(f"{y}-12-31").dayofyear == 366 else 365


def robust_growth(annual, method, last_year):
    years = sorted(annual.index.tolist())

    if method == "zero":
        return 0.0

    if method == "last":
        if last_year - 1 in annual.index and last_year in annual.index:
            return annual.loc[last_year] / annual.loc[last_year - 1] - 1
        return 0.0

    if method == "cagr2":
        y0 = last_year - 2
        if y0 in annual.index and last_year in annual.index:
            return (annual.loc[last_year] / annual.loc[y0]) ** (1 / 2) - 1
        return 0.0

    if method == "cagr3":
        y0 = last_year - 3
        if y0 in annual.index and last_year in annual.index:
            return (annual.loc[last_year] / annual.loc[y0]) ** (1 / 3) - 1
        return 0.0

    if method == "postcovid":
        valid = [y for y in [2020, 2021, 2022] if y in annual.index]
        if len(valid) >= 2:
            y0, y1 = valid[0], valid[-1]
            return (annual.loc[y1] / annual.loc[y0]) ** (1 / (y1 - y0)) - 1
        return 0.0

    if method == "covid_recovery":
        if 2021 in annual.index and 2022 in annual.index:
            return annual.loc[2022] / annual.loc[2021] - 1
        return 0.0

    return 0.0


def make_profile(train_df, target, profile_years, profile_kind):
    df = train_df.copy()
    df["year"] = df["Date"].dt.year
    df["doy"] = df["Date"].dt.dayofyear.clip(upper=365)
    df["month"] = df["Date"].dt.month
    df["day"] = df["Date"].dt.day
    df["dow"] = df["Date"].dt.dayofweek

    last_year = df["year"].max()
    recent = df[df["year"] >= last_year - profile_years + 1].copy()

    if len(recent) < 100:
        recent = df.copy()

    if profile_kind == "doy":
        prof = recent.groupby("doy")[target].mean()
        prof = prof / prof.mean()
        key = "doy"

    elif profile_kind == "month_day":
        recent["md"] = recent["month"].astype(str) + "-" + recent["day"].astype(str)
        prof = recent.groupby("md")[target].mean()
        prof = prof / prof.mean()
        key = "md"

    elif profile_kind == "month_dow":
        recent["mdow"] = recent["month"].astype(str) + "-" + recent["dow"].astype(str)
        prof = recent.groupby("mdow")[target].mean()
        prof = prof / prof.mean()
        key = "mdow"

    else:
        prof = recent.groupby("doy")[target].mean()
        prof = prof / prof.mean()
        key = "doy"

    return prof, key


def forecast_config(train_df, future_dates, target, cfg):
    train_df = train_df.copy()
    future_dates = pd.to_datetime(pd.Series(future_dates)).sort_values().reset_index(drop=True)

    train_df["year"] = train_df["Date"].dt.year
    train_df["doy"] = train_df["Date"].dt.dayofyear.clip(upper=365)

    annual = train_df.groupby("year")[target].sum()
    last_year = int(train_df["year"].max())
    last_total = float(annual.loc[last_year])

    growth = robust_growth(annual, cfg["growth"], last_year)
    growth = float(np.clip(growth, cfg["gmin"], cfg["gmax"]))

    prof, key = make_profile(
        train_df=train_df,
        target=target,
        profile_years=cfg["profile_years"],
        profile_kind=cfg["profile_kind"]
    )

    hist = train_df.set_index("Date")[target].to_dict()
    preds = []

    for dt in future_dates:
        y = dt.year
        doy = min(dt.dayofyear, 365)
        md = str(dt.month) + "-" + str(dt.day)
        mdow = str(dt.month) + "-" + str(dt.dayofweek)

        if key == "doy":
            season = prof.get(doy, 1.0)
        elif key == "md":
            season = prof.get(md, 1.0)
        elif key == "mdow":
            season = prof.get(mdow, 1.0)
        else:
            season = 1.0

        years_ahead = y - last_year
        annual_pred = last_total * ((1 + growth) ** years_ahead)
        profile_pred = annual_pred / get_days_in_year(y) * season

        d1 = pd.Timestamp(dt) - pd.DateOffset(years=1)
        d2 = pd.Timestamp(dt) - pd.DateOffset(years=2)
        d3 = pd.Timestamp(dt) - pd.DateOffset(years=3)

        sly1 = hist.get(pd.Timestamp(d1), np.nan)
        sly2 = hist.get(pd.Timestamp(d2), np.nan)
        sly3 = hist.get(pd.Timestamp(d3), np.nan)

        candidates = []

        if not pd.isna(sly1):
            candidates.append(sly1 * (1 + growth))
        if not pd.isna(sly2):
            candidates.append(sly2 * ((1 + growth) ** 2))
        if not pd.isna(sly3):
            candidates.append(sly3 * ((1 + growth) ** 3))

        if len(candidates) == 0:
            same_year_pred = profile_pred
        else:
            same_year_pred = np.average(
                candidates,
                weights=np.array([0.60, 0.30, 0.10])[:len(candidates)]
            )

        pred = cfg["w_profile"] * profile_pred + (1 - cfg["w_profile"]) * same_year_pred

        if cfg["level"] == "last_year":
            level_base = train_df[train_df["year"] == last_year][target].mean()
            pred = cfg["level_blend"] * pred + (1 - cfg["level_blend"]) * level_base * season

        elif cfg["level"] == "postcovid_mean":
            post = train_df[train_df["year"] >= max(2020, last_year - 2)]
            level_base = post[target].mean()
            pred = cfg["level_blend"] * pred + (1 - cfg["level_blend"]) * level_base * season

        pred = max(float(pred), 0.0)
        preds.append(pred)

        hist[pd.Timestamp(dt)] = pred

    return np.array(preds)

In [7]:
configs_fast = []

profile_years_list = [1, 2, 3]
profile_kind_list = ["doy", "month_day", "month_dow"]

growth_list = [
    "zero",
    "last",
    "cagr2",
    "postcovid",
    "covid_recovery"
]

w_profile_list = [0.0, 0.35, 0.65, 1.0]

level_list = ["last_year", "postcovid_mean"]
level_blend_list = [0.75, 0.90, 1.00]

for profile_years in profile_years_list:
    for profile_kind in profile_kind_list:
        for growth in growth_list:
            for w_profile in w_profile_list:
                for level in level_list:
                    for level_blend in level_blend_list:
                        configs_fast.append({
                            "profile_years": profile_years,
                            "profile_kind": profile_kind,
                            "growth": growth,
                            "w_profile": w_profile,
                            "level": level,
                            "level_blend": level_blend,
                            "gmin": -0.10,
                            "gmax": 0.14,
                        })

print("Fast configs:", len(configs_fast))

Fast configs: 1080


In [8]:
try:
    from tqdm.auto import tqdm
except:
    tqdm = lambda x: x

def validate_single_year(df, target, cfg, val_year):
    train_df = df[df["Date"] < pd.Timestamp(f"{val_year}-01-01")].copy()

    valid_df = df[
        (df["Date"] >= pd.Timestamp(f"{val_year}-01-01")) &
        (df["Date"] <= pd.Timestamp(f"{val_year}-12-31"))
    ].copy()

    if len(train_df) < 365 * 3 or len(valid_df) < 300:
        return np.inf

    try:
        pred = forecast_config(train_df, valid_df["Date"], target, cfg)

        if not np.all(np.isfinite(pred)):
            return np.inf

        return rmse_score(valid_df[target].values, pred)

    except:
        return np.inf


all_results = {}

for target in ["Revenue", "COGS"]:
    print("\n" + "=" * 70)
    print("Fast validating:", target)
    print("=" * 70)

    stage1 = []

    for cfg in tqdm(configs_fast):
        score_2022 = validate_single_year(
            df=sales_clean,
            target=target,
            cfg=cfg,
            val_year=2022
        )

        if np.isfinite(score_2022):
            stage1.append((score_2022, cfg))

    stage1 = sorted(stage1, key=lambda x: x[0])

    print("Stage 1 valid configs:", len(stage1))
    print("Best 2022 RMSE:", stage1[0][0])

    top_stage1 = stage1[:80]

    stage2 = []

    for score_2022, cfg in tqdm(top_stage1):
        s2020 = validate_single_year(sales_clean, target, cfg, 2020)
        s2021 = validate_single_year(sales_clean, target, cfg, 2021)
        s2022 = score_2022

        final_score = np.average(
            [s2020, s2021, s2022],
            weights=[0.25, 0.75, 3.00]
        )

        if np.isfinite(final_score):
            stage2.append((final_score, cfg, s2020, s2021, s2022))

    stage2 = sorted(stage2, key=lambda x: x[0])

    all_results[target] = [(x[0], x[1]) for x in stage2]

    print("\nTop 10 configs for", target)

    for rank, row in enumerate(stage2[:10], 1):
        final_score, cfg, s2020, s2021, s2022 = row

        print(
            rank,
            "| final =", round(final_score, 2),
            "| 2020 =", round(s2020, 2),
            "| 2021 =", round(s2021, 2),
            "| 2022 =", round(s2022, 2),
            "| cfg =", cfg
        )


Fast validating: Revenue


  0%|          | 0/1080 [00:00<?, ?it/s]

Stage 1 valid configs: 1080
Best 2022 RMSE: 957131.8599647365


  0%|          | 0/80 [00:00<?, ?it/s]


Top 10 configs for Revenue
1 | final = 950041.47 | 2020 = 904016.32 | 2021 = 898409.71 | 2022 = 966784.84 | cfg = {'profile_years': 3, 'profile_kind': 'doy', 'growth': 'last', 'w_profile': 0.65, 'level': 'last_year', 'level_blend': 0.75, 'gmin': -0.1, 'gmax': 0.14}
2 | final = 950433.03 | 2020 = 946796.5 | 2021 = 901728.09 | 2022 = 962912.31 | cfg = {'profile_years': 3, 'profile_kind': 'month_day', 'growth': 'last', 'w_profile': 1.0, 'level': 'last_year', 'level_blend': 0.75, 'gmin': -0.1, 'gmax': 0.14}
3 | final = 951483.77 | 2020 = 940251.21 | 2021 = 905164.05 | 2022 = 963999.75 | cfg = {'profile_years': 3, 'profile_kind': 'month_day', 'growth': 'last', 'w_profile': 1.0, 'level': 'last_year', 'level_blend': 0.9, 'gmin': -0.1, 'gmax': 0.14}
4 | final = 952275.0 | 2020 = 1019710.11 | 2021 = 908147.74 | 2022 = 957687.22 | cfg = {'profile_years': 3, 'profile_kind': 'month_day', 'growth': 'zero', 'w_profile': 1.0, 'level': 'last_year', 'level_blend': 1.0, 'gmin': -0.1, 'gmax': 0.14}
5 | 

  0%|          | 0/1080 [00:00<?, ?it/s]

Stage 1 valid configs: 1080
Best 2022 RMSE: 764988.6911537446


  0%|          | 0/80 [00:00<?, ?it/s]


Top 10 configs for COGS
1 | final = 764523.97 | 2020 = 807270.37 | 2021 = 748416.29 | 2022 = 764988.69 | cfg = {'profile_years': 3, 'profile_kind': 'month_dow', 'growth': 'last', 'w_profile': 0.0, 'level': 'last_year', 'level_blend': 0.75, 'gmin': -0.1, 'gmax': 0.14}
2 | final = 770467.82 | 2020 = 1049864.11 | 2021 = 698281.35 | 2022 = 765231.41 | cfg = {'profile_years': 3, 'profile_kind': 'month_dow', 'growth': 'postcovid', 'w_profile': 0.35, 'level': 'last_year', 'level_blend': 1.0, 'gmin': -0.1, 'gmax': 0.14}
3 | final = 770570.76 | 2020 = 1114210.98 | 2021 = 678352.29 | 2022 = 764988.69 | cfg = {'profile_years': 3, 'profile_kind': 'month_dow', 'growth': 'postcovid', 'w_profile': 0.0, 'level': 'last_year', 'level_blend': 0.75, 'gmin': -0.1, 'gmax': 0.14}
4 | final = 772883.42 | 2020 = 761410.95 | 2021 = 807315.62 | 2022 = 765231.41 | cfg = {'profile_years': 3, 'profile_kind': 'month_dow', 'growth': 'last', 'w_profile': 0.35, 'level': 'last_year', 'level_blend': 1.0, 'gmin': -0.1, '

In [9]:
def predict_ensemble_top(df, future_dates, target, top_k=15):
    rows = all_results[target][:top_k]

    preds = []
    weights = []

    for score, cfg in rows:
        pred = forecast_config(df, future_dates, target, cfg)

        w = 1.0 / ((score + 1e-9) ** 1.25)

        preds.append(pred)
        weights.append(w)

    preds = np.vstack(preds)

    weights = np.array(weights)
    weights = weights / weights.sum()

    final_pred = np.average(preds, axis=0, weights=weights)

    return final_pred


preds = {}

for target in ["Revenue", "COGS"]:
    pred = predict_ensemble_top(
        df=sales_clean,
        future_dates=sample["Date"],
        target=target,
        top_k=15
    )

    post = sales_clean[sales_clean["Date"] >= "2020-01-01"][target]

    lo = post.quantile(0.001) * 0.70
    hi = post.quantile(0.999) * 1.25

    pred = np.clip(pred, lo, hi)

    preds[target] = pred

    print(target)
    print("first:", pred[:5])
    print("last :", pred[-5:])

Revenue
first: [2440654.15783326 1191608.81129927  756014.16540596 1130623.2439231
 1240544.50594572]
last : [4376058.30657061 5914414.17247427 7331648.27919101 6509838.6532964
 5999241.73700023]
COGS
first: [2443758.03134165 1501934.95867543  763345.65814411 1061054.55520313
  986214.14979987]
last : [4464791.4211876  5571978.83769334 6104922.54030872 5692858.07793647
 5197381.47585154]


In [10]:
sub = sample[["Date"]].copy()
sub["Revenue"] = preds["Revenue"]
sub["COGS"] = preds["COGS"]

ratio = sub["COGS"] / (sub["Revenue"] + 1e-9)

too_high = ratio > 1.08
sub.loc[too_high, "COGS"] = sub.loc[too_high, "Revenue"] * 1.01

too_low = ratio < 0.45
sub.loc[too_low, "COGS"] = sub.loc[too_low, "Revenue"] * 0.55

sub["Revenue"] = sub["Revenue"].round(2)
sub["COGS"] = sub["COGS"].round(2)

OUT_PATH = "/content/submission_v2.csv"
sub.to_csv(OUT_PATH, index=False)

print("Saved:", OUT_PATH)
print(sub.shape)
display(sub.head())
display(sub.tail())

print(sub.describe())
print(sub.isna().sum())

Saved: /content/submission_v2.csv
(548, 3)


,Date,Revenue,COGS
0,2023-01-01,2440654.16,2443758.03
1,2023-01-02,1191608.81,1203524.90
2,2023-01-03,756014.17,763345.66
3,2023-01-04,1130623.24,1061054.56
4,2023-01-05,1240544.51,986214.15


,Date,Revenue,COGS
543,2024-06-27,4376058.31,4464791.42
544,2024-06-28,5914414.17,5571978.84
545,2024-06-29,7331648.28,6104922.54
546,2024-06-30,6509838.65,5692858.08
547,2024-07-01,5999241.74,5197381.48


                      Date       Revenue          COGS
count                  548  5.480000e+02  5.480000e+02
mean   2023-10-01 12:00:00  3.708947e+06  3.168025e+06
min    2023-01-01 00:00:00  7.073391e+05  7.144125e+05
25%    2023-05-17 18:00:00  2.256511e+06  1.977602e+06
50%    2023-10-01 12:00:00  3.342269e+06  3.112250e+06
75%    2024-02-15 06:00:00  4.896380e+06  4.106617e+06
max    2024-07-01 00:00:00  1.060681e+07  8.259103e+06
std                    NaN  1.878206e+06  1.427531e+06
Date       0
Revenue    0
COGS       0
dtype: int64


In [11]:
import os
import pandas as pd
import numpy as np

BEST_PATH = "/content/submission_v2.csv"


assert os.path.exists(BEST_PATH), f"Không thấy file: {BEST_PATH}"

base = pd.read_csv(BEST_PATH)
base["Date"] = pd.to_datetime(base["Date"])

print(base.shape)
display(base.head())
display(base.tail())
print(base[["Revenue", "COGS"]].describe())

(548, 3)


,Date,Revenue,COGS
0,2023-01-01,2440654.16,2443758.03
1,2023-01-02,1191608.81,1203524.90
2,2023-01-03,756014.17,763345.66
3,2023-01-04,1130623.24,1061054.56
4,2023-01-05,1240544.51,986214.15


,Date,Revenue,COGS
543,2024-06-27,4376058.31,4464791.42
544,2024-06-28,5914414.17,5571978.84
545,2024-06-29,7331648.28,6104922.54
546,2024-06-30,6509838.65,5692858.08
547,2024-07-01,5999241.74,5197381.48


            Revenue          COGS
count  5.480000e+02  5.480000e+02
mean   3.708947e+06  3.168025e+06
std    1.878206e+06  1.427531e+06
min    7.073391e+05  7.144125e+05
25%    2.256511e+06  1.977602e+06
50%    3.342269e+06  3.112250e+06
75%    4.896380e+06  4.106617e+06
max    1.060681e+07  8.259103e+06


In [12]:
import os, glob, zipfile, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)

BEST_SCALE = 1.18

V2_PATH_CANDIDATES = [
    "/content/submission_v2.csv",
    "/content/submission_covid_ensemble_v2.csv",
    "/content/submission.csv",
]

V2_PATH = None
for p in V2_PATH_CANDIDATES:
    if os.path.exists(p):
        V2_PATH = p
        break

assert V2_PATH is not None, "Không tìm thấy file V2 backbone. Hãy chạy lại cell tạo submission_v2.csv trước."

v2_base = pd.read_csv(V2_PATH)
v2_base["Date"] = pd.to_datetime(v2_base["Date"])
v2_base = v2_base.sort_values("Date").reset_index(drop=True)

v2_scaled_118 = v2_base.copy()
v2_scaled_118["Revenue"] = (v2_scaled_118["Revenue"] * BEST_SCALE).round(2)
v2_scaled_118["COGS"] = (v2_scaled_118["COGS"] * BEST_SCALE).round(2)

OUT_BEST_SCALE = "/content/submission_v11_best_scale_118.csv"
v2_scaled_118.to_csv(OUT_BEST_SCALE, index=False)

print("Loaded V2:", V2_PATH)
print("Saved best scale backbone:", OUT_BEST_SCALE)
print("BEST_SCALE =", BEST_SCALE)
display(v2_scaled_118.head())
display(v2_scaled_118.tail())
print(v2_scaled_118[["Revenue", "COGS"]].describe())

Loaded V2: /content/submission_v2.csv
Saved best scale backbone: /content/submission_v11_best_scale_118.csv
BEST_SCALE = 1.18


,Date,Revenue,COGS
0,2023-01-01,2879971.91,2883634.48
1,2023-01-02,1406098.40,1420159.38
2,2023-01-03,892096.72,900747.88
3,2023-01-04,1334135.42,1252044.38
4,2023-01-05,1463842.52,1163732.70


,Date,Revenue,COGS
543,2024-06-27,5163748.81,5268453.88
544,2024-06-28,6979008.72,6574935.03
545,2024-06-29,8651344.97,7203808.60
546,2024-06-30,7681609.61,6717572.53
547,2024-07-01,7079105.25,6132910.15


            Revenue          COGS
count  5.480000e+02  5.480000e+02
mean   4.376557e+06  3.738270e+06
std    2.216284e+06  1.684487e+06
min    8.346601e+05  8.430067e+05
25%    2.662684e+06  2.333570e+06
50%    3.943878e+06  3.672455e+06
75%    5.777728e+06  4.845808e+06
max    1.251603e+07  9.745742e+06


In [13]:
import os, glob, zipfile
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error


def rmse_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() == 0:
        return np.inf
    return np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))


def clean_sales_strong_for_v11(df):
    df = df.copy()
    df = df[["Date", "Revenue", "COGS"]]
    df = df.groupby("Date", as_index=False)[["Revenue", "COGS"]].sum()

    full_dates = pd.date_range(df["Date"].min(), df["Date"].max(), freq="D")
    df = df.set_index("Date").reindex(full_dates)
    df.index.name = "Date"

    for col in ["Revenue", "COGS"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].interpolate(limit_direction="both")
        df[col] = df[col].fillna(df[col].median())

        roll_med = df[col].rolling(45, center=True, min_periods=10).median()
        resid = df[col] - roll_med
        mad = resid.abs().rolling(45, center=True, min_periods=10).median()

        lower = roll_med - 7.5 * mad
        upper = roll_med + 7.5 * mad
        mask = mad.notna() & (mad > 1)

        df.loc[mask, col] = np.minimum(
            np.maximum(df.loc[mask, col], lower.loc[mask]),
            upper.loc[mask]
        )

    return df.reset_index()

if "sales_clean" not in globals() or "sample" not in globals():
    ZIP_PATH = "/content/datathon-2026-round-1.zip"
    EXTRACT_DIR = "/content/datathon_data"

    if os.path.exists(ZIP_PATH) and not os.path.exists(EXTRACT_DIR):
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            z.extractall(EXTRACT_DIR)

    sales_candidates = glob.glob("/content/**/sales.csv", recursive=True)
    sample_candidates = glob.glob("/content/**/sample_submission.csv", recursive=True)

    assert len(sales_candidates) > 0, "Không tìm thấy sales.csv"
    assert len(sample_candidates) > 0, "Không tìm thấy sample_submission.csv"

    sales = pd.read_csv(sales_candidates[0])
    sample = pd.read_csv(sample_candidates[0])
    sales["Date"] = pd.to_datetime(sales["Date"])
    sample["Date"] = pd.to_datetime(sample["Date"])
    sales = sales.sort_values("Date").reset_index(drop=True)
    sample = sample.sort_values("Date").reset_index(drop=True)
    sales_clean = clean_sales_strong_for_v11(sales)
else:
    sales_clean = sales_clean.copy()
    sample = sample.copy()
    sales_clean["Date"] = pd.to_datetime(sales_clean["Date"])
    sample["Date"] = pd.to_datetime(sample["Date"])

sales_clean = sales_clean.sort_values("Date").reset_index(drop=True)
sample = sample.sort_values("Date").reset_index(drop=True)

print("sales_clean:", sales_clean.shape, sales_clean["Date"].min(), "->", sales_clean["Date"].max())
print("sample:", sample.shape, sample["Date"].min(), "->", sample["Date"].max())
print(sales_clean.isna().sum())

sales_clean: (3833, 3) 2012-07-04 00:00:00 -> 2022-12-31 00:00:00
sample: (548, 3) 2023-01-01 00:00:00 -> 2024-07-01 00:00:00
Date       0
Revenue    0
COGS       0
dtype: int64


In [14]:
try:
    import xgboost as xgb
    print("xgboost version:", xgb.__version__)
except Exception:
    !pip -q install xgboost
    import xgboost as xgb
    print("xgboost version:", xgb.__version__)

xgboost version: 3.2.0


In [15]:
def make_ts_features_v11(df, target):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

    out = pd.DataFrame({"Date": df["Date"]})
    d = out["Date"]

    out["year"] = d.dt.year
    out["month"] = d.dt.month
    out["day"] = d.dt.day
    out["dow"] = d.dt.dayofweek
    out["doy"] = d.dt.dayofyear.clip(upper=365)
    out["week"] = d.dt.isocalendar().week.astype(int)
    out["quarter"] = d.dt.quarter

    out["is_weekend"] = (out["dow"] >= 5).astype(int)
    out["is_month_start"] = d.dt.is_month_start.astype(int)
    out["is_month_end"] = d.dt.is_month_end.astype(int)
    out["is_quarter_start"] = d.dt.is_quarter_start.astype(int)
    out["is_quarter_end"] = d.dt.is_quarter_end.astype(int)

    min_date = df["Date"].min()
    out["t"] = (df["Date"] - min_date).dt.days
    out["t2"] = out["t"] ** 2

    for k in range(1, 9):
        out[f"sin_y_{k}"] = np.sin(2 * np.pi * k * out["doy"] / 365.0)
        out[f"cos_y_{k}"] = np.cos(2 * np.pi * k * out["doy"] / 365.0)

    for k in range(1, 4):
        out[f"sin_w_{k}"] = np.sin(2 * np.pi * k * out["dow"] / 7.0)
        out[f"cos_w_{k}"] = np.cos(2 * np.pi * k * out["dow"] / 7.0)

    out["is_covid_2020"] = ((d >= "2020-01-01") & (d <= "2020-12-31")).astype(int)
    out["is_covid_2021"] = ((d >= "2021-01-01") & (d <= "2021-12-31")).astype(int)
    out["is_post_covid"] = (d >= "2022-01-01").astype(int)
    out["post_covid_t"] = np.maximum(0, (d - pd.Timestamp("2022-01-01")).dt.days)

    out["is_nov_dec"] = out["month"].isin([11, 12]).astype(int)
    out["is_jan"] = (out["month"] == 1).astype(int)
    out["is_mid_year"] = out["month"].isin([6, 7]).astype(int)

    s = df.set_index("Date")[target].sort_index()

    for lag in [1, 7, 14, 28, 56, 91, 182, 365]:
        out[f"lag_{lag}"] = out["Date"].map(s.shift(lag))

    for win in [7, 14, 28, 56, 91, 182, 365]:
        shifted = s.shift(1)
        out[f"roll_mean_{win}"] = out["Date"].map(shifted.rolling(win, min_periods=max(3, win // 4)).mean())
        out[f"roll_std_{win}"] = out["Date"].map(shifted.rolling(win, min_periods=max(3, win // 4)).std())
        out[f"roll_min_{win}"] = out["Date"].map(shifted.rolling(win, min_periods=max(3, win // 4)).min())
        out[f"roll_max_{win}"] = out["Date"].map(shifted.rolling(win, min_periods=max(3, win // 4)).max())

    out["lag_365_ratio"] = out["lag_365"] / (out["roll_mean_365"] + 1e-9)
    out["lag_7_ratio"] = out["lag_7"] / (out["roll_mean_28"] + 1e-9)

    out[target] = df[target].values
    return out

In [16]:
def xgb_recursive_forecast_v11(train_df, future_dates, target, params):
    hist = train_df[["Date", target]].copy()
    hist["Date"] = pd.to_datetime(hist["Date"])
    hist = hist.sort_values("Date").reset_index(drop=True)

    feat = make_ts_features_v11(hist, target)
    feat = feat[feat["Date"] >= hist["Date"].min() + pd.Timedelta(days=370)].copy()

    feature_cols = [c for c in feat.columns if c not in ["Date", target]]

    X = feat[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y = np.log1p(feat[target].values)

    years = feat["Date"].dt.year.values
    weights = np.ones(len(feat))
    weights[years >= 2018] = 1.2
    weights[years >= 2019] = 1.5
    weights[years >= 2020] = 2.2
    weights[years >= 2022] = 3.2

    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        random_state=SEED,
        tree_method="hist",
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        min_child_weight=params["min_child_weight"],
        reg_lambda=params["reg_lambda"],
        reg_alpha=params["reg_alpha"],
        max_bin=params["max_bin"],
        n_jobs=-1,
    )

    model.fit(X, y, sample_weight=weights)

    preds = []
    for dt in pd.to_datetime(pd.Series(future_dates)):
        temp = pd.concat([
            hist,
            pd.DataFrame({"Date": [dt], target: [np.nan]})
        ], ignore_index=True)

        temp_feat = make_ts_features_v11(temp, target).iloc[[-1]]
        for c in feature_cols:
            if c not in temp_feat.columns:
                temp_feat[c] = 0

        X_one = temp_feat[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
        pred = np.expm1(model.predict(X_one)[0])
        pred = max(float(pred), 0.0)
        preds.append(pred)

        hist = pd.concat([
            hist,
            pd.DataFrame({"Date": [dt], target: [pred]})
        ], ignore_index=True)

    return np.array(preds)

In [17]:
xgb_params_grid = [
    dict(n_estimators=500, max_depth=2, learning_rate=0.025, subsample=0.90, colsample_bytree=0.90, min_child_weight=5, reg_lambda=10.0, reg_alpha=0.2, max_bin=256),
    dict(n_estimators=700, max_depth=3, learning_rate=0.018, subsample=0.85, colsample_bytree=0.85, min_child_weight=4, reg_lambda=8.0, reg_alpha=0.3, max_bin=256),
    dict(n_estimators=900, max_depth=2, learning_rate=0.015, subsample=0.90, colsample_bytree=0.80, min_child_weight=6, reg_lambda=12.0, reg_alpha=0.5, max_bin=256),
    dict(n_estimators=600, max_depth=4, learning_rate=0.020, subsample=0.80, colsample_bytree=0.80, min_child_weight=4, reg_lambda=10.0, reg_alpha=0.5, max_bin=256),
]

xgb_results = {}
xgb_preds = {}

for target in ["Revenue", "COGS"]:
    print("" + "=" * 70)
    print("Target:", target)
    print("=" * 70)

    train_v = sales_clean[sales_clean["Date"] < "2022-01-01"].copy()
    valid_v = sales_clean[(sales_clean["Date"] >= "2022-01-01") & (sales_clean["Date"] <= "2022-12-31")].copy()

    rows = []
    for i, params in enumerate(xgb_params_grid, 1):
        print("Params", i, params)
        pred_valid = xgb_recursive_forecast_v11(train_v, valid_v["Date"], target, params)
        score = rmse_score(valid_v[target].values, pred_valid)
        rows.append((score, params))
        print("Valid 2022 RMSE:", round(score, 2))

    rows = sorted(rows, key=lambda x: x[0])
    xgb_results[target] = rows

    best_params = rows[0][1]
    print("Best params:", best_params)

    pred_test = xgb_recursive_forecast_v11(sales_clean, sample["Date"], target, best_params)
    ref = sales_clean[sales_clean["Date"] >= "2020-01-01"][target]
    pred_test = np.clip(pred_test, ref.quantile(0.001) * 0.70, ref.quantile(0.999) * 1.35)
    xgb_preds[target] = pred_test

    print("Test pred first:", pred_test[:5])
    print("Test pred last :", pred_test[-5:])

Target: Revenue
Params 1 {'n_estimators': 500, 'max_depth': 2, 'learning_rate': 0.025, 'subsample': 0.9, 'colsample_bytree': 0.9, 'min_child_weight': 5, 'reg_lambda': 10.0, 'reg_alpha': 0.2, 'max_bin': 256}
Valid 2022 RMSE: 1033651.33
Params 2 {'n_estimators': 700, 'max_depth': 3, 'learning_rate': 0.018, 'subsample': 0.85, 'colsample_bytree': 0.85, 'min_child_weight': 4, 'reg_lambda': 8.0, 'reg_alpha': 0.3, 'max_bin': 256}
Valid 2022 RMSE: 991890.62
Params 3 {'n_estimators': 900, 'max_depth': 2, 'learning_rate': 0.015, 'subsample': 0.9, 'colsample_bytree': 0.8, 'min_child_weight': 6, 'reg_lambda': 12.0, 'reg_alpha': 0.5, 'max_bin': 256}
Valid 2022 RMSE: 1031894.94
Params 4 {'n_estimators': 600, 'max_depth': 4, 'learning_rate': 0.02, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 4, 'reg_lambda': 10.0, 'reg_alpha': 0.5, 'max_bin': 256}
Valid 2022 RMSE: 971732.53
Best params: {'n_estimators': 600, 'max_depth': 4, 'learning_rate': 0.02, 'subsample': 0.8, 'colsample_bytree'

In [18]:
blend_weights = [0.15]
created_v11_files = []

for w_xgb in blend_weights:
    sub = sample[["Date"]].copy()

    sub["Revenue"] = (1 - w_xgb) * v2_scaled_118["Revenue"].values + w_xgb * xgb_preds["Revenue"]
    sub["COGS"] = (1 - w_xgb) * v2_scaled_118["COGS"].values + w_xgb * xgb_preds["COGS"]

    sub["Revenue"] = sub["Revenue"].round(2)
    sub["COGS"] = sub["COGS"].round(2)

    out_path = f"/content/submission_Finally.csv"
    sub.to_csv(out_path, index=False)
    created_v11_files.append(out_path)


In [ ]:
BEST_PATH = "/content/submission_Finally.csv"

best = pd.read_csv(BEST_PATH)
best["Date"] = pd.to_datetime(best["Date"])

created = []

sub = best[["Date", "Revenue", "COGS"]].copy()
sub["Revenue"] *= 1.050
sub["COGS"] *= 1.050
sub["Revenue"] = sub["Revenue"].round(2)
sub["COGS"] = sub["COGS"].round(2)
path = "/content/submission.csv"
sub.to_csv(path, index=False)
created.append(path)